# 06 · Code safari: reading real library code

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 1 | about 30 minutes live | [05 · Polymorphism, duck typing and protocols](05_polymorphism_duck_typing_protocols.ipynb) | [07 · Type hints, docstrings and dataclasses](07_type_hints_docstrings_dataclasses.ipynb) |

Today you wrote classes with inheritance, properties, dunder methods, abstract base classes and
protocols. The same ideas hold up the libraries you use every day. Reading their source is the fastest
way to get comfortable with code written by other people, and it is less scary than it sounds: you
already know every concept you will meet below.

> **Learning goals.** By the end you can:
>
> - recognise inheritance, mixins, dunder methods and the fit/transform conventions in scikit-learn's source
> - use `inspect` and your editor to find where a method is really defined
> - explain to a colleague why a library class is designed the way it is

**Before you start:** notebooks 01 to 05.

**How we work in the session (jigsaw).** Each pair takes one station (1 to 5) for about 12 minutes:
run the cells, answer the questions, then open the answers and compare. Afterwards every pair explains
its station to the group in 2 minutes. Station 6 is a **Challenge** for experienced programmers.
Afterwards, work through the remaining stations at your own pace.

---

## A small helper for reading source code

`inspect.getsource(obj)` returns the source code of a class, function or method from an installed
library. Library methods have long docstrings, so the helper below hides them and can keep only the
lines you care about. You do not need to understand how the helper works; read the outputs.

In [ ]:
import ast
import inspect
import re
import textwrap

import sklearn

print("scikit-learn version:", sklearn.__version__)


def show(obj, grep: str | None = None, last: int | None = None) -> None:
    """Print the source of `obj` without its docstring.

    grep: only keep lines matching this regular expression (plus the `def` line).
    last: only keep the last N lines.
    """
    src = textwrap.dedent(inspect.getsource(obj))
    lines = src.splitlines()
    node = ast.parse(src).body[0]
    if ast.get_docstring(node):
        doc = node.body[0]
        indent = " " * doc.col_offset
        lines[doc.lineno - 1 : doc.end_lineno] = [f'{indent}"""(docstring hidden)"""']
    if grep:
        lines = [
            line for line in lines if re.search(grep, line) or line.lstrip().startswith("def ")
        ]
        lines.insert(0, "# (only matching lines shown)")
    if last:
        lines = ["    ..."] + lines[-last:]
    # decorators wrap some methods; unwrap to find the file the code really lives in
    path = inspect.getsourcefile(inspect.unwrap(obj))
    short_path = re.split(r"site-packages/|/lib/python3\.\d+/", path)[-1]
    print(f"# {short_path}")
    print("\n".join(lines))

We also need a small, real dataset to try things on. We use the churn data with two numeric columns.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../data/raw/telco_churn.csv")
customers = pd.read_csv(DATA_PATH)
X = customers[["tenure", "MonthlyCharges"]]
y = (customers["Churn"] == "Yes").astype(int)

---

## Station 1 · `DummyClassifier`: inheritance and the fit convention

`DummyClassifier` is scikit-learn's version of the baselines you wrote in notebook 02 (for example
`strategy="most_frequent"` is our `MajorityClassModel`).

In [ ]:
from sklearn.dummy import DummyClassifier

print(inspect.getsource(DummyClassifier).splitlines()[0])
print()
print("MRO:", [cls.__name__ for cls in DummyClassifier.__mro__])

In [ ]:
# Only the lines of fit() that store something ending in "_" or return
show(DummyClassifier.fit, grep=r"self\.\w+_\b.*=|return self\b")

In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(X, y)
print("Attributes set by fit:", [name for name in vars(baseline) if name.endswith("_")])
print("Accuracy of predicting 'nobody churns':", round(baseline.score(X, y), 3))

In [ ]:
show(DummyClassifier.score)

**Questions**

1. Which classes does `DummyClassifier` inherit from directly? Which of them are mixins?
2. What does `fit` return, and what does that allow you to write in one line?
3. How can you tell, just from the name, that `classes_` was learned from data and was not passed by you?
4. `ClassifierMixin` already provides `score`. Why does `DummyClassifier` define its own, and what does
   `super().score(...)` do there?

<details><summary><b>Show answers</b></summary>

1. `MultiOutputMixin`, `ClassifierMixin` and `BaseEstimator` (the first line of the class). The two
   `...Mixin` classes each add one capability; `BaseEstimator` provides `get_params`, `set_params` and
   the nice `repr`. The longer MRO also lists the parents of `BaseEstimator`.
2. `self`. That is what makes `DummyClassifier(...).fit(X, y).predict(X)` and `.fit(X, y).score(X, y)`
   work (method chaining). It is the same convention you used in notebooks 02 and 04.
3. The trailing underscore. scikit-learn's convention: attributes learned in `fit` end with `_`;
   constructor parameters (like `strategy`) do not.
4. It overrides `score` to allow `X=None` (a dummy model ignores the features anyway) and then calls
   the parent's implementation with `super()`: extend, don't copy. Exactly the pattern from notebook 02.
</details>

---

## Station 2 · `TransformerMixin.fit_transform`: a mixin and a template method

In [ ]:
from sklearn.base import TransformerMixin

show(TransformerMixin.fit_transform, last=5)

In [ ]:
from sklearn.preprocessing import StandardScaler

print(
    "StandardScaler.fit_transform is TransformerMixin.fit_transform:",
    StandardScaler.fit_transform is TransformerMixin.fit_transform,
)
print(
    "Classes that define fit/transform:",
    [cls.__name__ for cls in StandardScaler.__mro__ if "transform" in vars(cls)],
)

**Questions**

1. `fit_transform` calls `self.fit(...)` and `.transform(...)`, but `TransformerMixin` defines neither.
   Why does this still work?
2. What must a class provide to get `fit_transform` for free?
3. Compare with the `Transformer` abstract base class from notebook 04. What is the difference between
   a mixin and an ABC here?

<details><summary><b>Show answers</b></summary>

1. Polymorphism. The method is only ever called on an object of a concrete class (e.g. `StandardScaler`)
   that does have `fit` and `transform`. Python looks them up on that object at call time.
2. `fit` (returning `self`) and `transform`. Nothing else.
3. Both provide `fit_transform` once for a whole family (a *template method*: a fixed recipe whose
   steps are filled in by subclasses). Our ABC also *forces* subclasses to implement `fit` and
   `transform` (instantiation fails otherwise). The mixin does not check anything; it relies on the
   convention, in duck-typing style. A missing `transform` only shows up when `fit_transform` is called.
</details>

---

## Station 3 · `Pipeline.__len__` and `__getitem__`: dunder methods and composition

In [ ]:
from sklearn.pipeline import Pipeline

show(Pipeline.__len__)
print()
show(Pipeline.__getitem__)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(StandardScaler(), LogisticRegression())
print("len(pipe):", len(pipe))
print("pipe[0]:", pipe[0])
print("pipe['logisticregression']:", pipe["logisticregression"])
print("type(pipe[:1]):", type(pipe[:1]).__name__)
print("steps:", pipe.steps)

**Questions**

1. Which special methods make `len(pipe)` and `pipe[0]` work?
2. `pipe[...]` accepts three kinds of keys. Which ones, and how does the code tell them apart?
3. Is a `Pipeline` a scaler? Is it a logistic regression? How is it related to them?

<details><summary><b>Show answers</b></summary>

1. `__len__` and `__getitem__` (notebook 03). Python translates `len(pipe)` into `pipe.__len__()` and
   `pipe[0]` into `pipe.__getitem__(0)`.
2. A slice (checked with `isinstance(ind, slice)`, returns a new `Pipeline`), an integer (indexes the
   list `self.steps`), and a step name (the integer lookup raises `TypeError` for a string, so the code
   falls back to `self.named_steps[ind]`). The last one is "try it and handle the error" duck typing.
3. Neither. A `Pipeline` *has* steps: it holds a list of `(name, estimator)` pairs and delegates to them.
   That is composition, the "has-a" relationship we will use on day 2 for `ChurnModel`. The pipeline
   itself is also an estimator (it has `fit` and `predict`), so a pipeline can be used anywhere a model
   can: the Composite pattern.
</details>

---

## Station 4 · `check_is_fitted`: how scikit-learn knows a model was fitted

In [ ]:
from sklearn.utils.validation import _is_fitted, check_is_fitted

show(check_is_fitted, last=7)
print()
show(_is_fitted, last=8)

In [ ]:
from sklearn.exceptions import NotFittedError

model = LogisticRegression()
try:
    model.predict(X)
except NotFittedError as exc:
    print(f"NotFittedError: {exc}")

model.fit(X, y)
print("After fit:", [name for name in vars(model) if name.endswith("_")])
check_is_fitted(model)  # no error now
print("check_is_fitted passed")

**Questions**

1. Without being told which attributes to look for, how does `_is_fitted` decide?
2. What is `__sklearn_is_fitted__` for?
3. In notebook 04 our classes kept a `_fitted` flag instead. Name one advantage of each approach.
4. What would go wrong if your own estimator set `self.threshold_ = 0.5` in `__init__`?

<details><summary><b>Show answers</b></summary>

1. It collects instance attributes (`vars(estimator)`) whose names end with `_` (but do not start with
   `__`). If there is at least one, the estimator counts as fitted. The naming convention from station 1
   is not just style: code relies on it.
2. A hook: a class can define this method to answer the question itself, and `_is_fitted` asks it
   first. It is the escape hatch for estimators whose state does not fit the naming convention.
3. The convention needs no extra code and works for every estimator automatically. An explicit flag is
   obvious to read, cannot be triggered by accident, and gives you full control over the error message.
4. The estimator would look fitted before `fit` was ever called, so the friendly `NotFittedError` would
   not appear and you would get a confusing error later (or silently wrong results).
</details>

---

## Station 5 · `pathlib.Path`: operator overloading in the standard library

Using `/` to join paths is not special syntax. It is a dunder method.

In [ ]:
from pathlib import Path, PurePath

show(PurePath.__truediv__)
print()
show(PurePath.__rtruediv__)

In [ ]:
data_dir = Path("..") / "data"
print(data_dir / "raw" / "telco_churn.csv")
print("../data" / Path("raw"))  # str / Path also works

**Questions**

1. Which special method does `Path("..") / "data"` call?
2. Why does `"../data" / Path("raw")` work, although `str` knows nothing about paths?
3. Why does the method return `NotImplemented` instead of raising an error itself?
4. Notebook 03 advised: only overload an operator when its meaning is obvious. Is `/` for paths a good
   use? Why?

<details><summary><b>Show answers</b></summary>

1. `__truediv__` (the method behind `/`).
2. Python first tries `str.__truediv__`, which does not exist, then falls back to the right operand's
   `__rtruediv__` ("reflected" division), which `PurePath` defines. Same mechanism as `__radd__` in
   notebook 03.
3. `NotImplemented` tells Python "I cannot handle this type, ask the other operand". Python only raises
   `TypeError` after both sides declined. Raising directly would prevent the other object from helping.
4. Yes: it reads like the path separator on Unix and in URLs, so nobody has to guess what it means. That
   is the test for any operator overloading.
</details>

---

## Station 6 · Challenge: why scikit-learn is strict about `__init__`

The scikit-learn developer guide says `__init__` must store each parameter unchanged, under the same
name, and do nothing else. Let us see why.

In [ ]:
from sklearn.base import BaseEstimator

show(BaseEstimator.get_params)
print()
show(BaseEstimator._get_param_names, grep=r"init = |init_signature = |return sorted")

In [ ]:
class ScaledThresholdModel(BaseEstimator):
    """Breaks the rule: stores a transformed value under a different name."""

    def __init__(self, threshold=0.5):
        self.cutoff = threshold * 100


try:
    ScaledThresholdModel(threshold=0.3).get_params()
except AttributeError as exc:
    print(f"AttributeError: {exc}")

**Questions**

1. Where does `get_params` get the list of parameter names from?
2. Why does `ScaledThresholdModel(...).get_params()` fail?
3. `clone`, `GridSearchCV` and `cross_val_score` all rely on `get_params` / `set_params`. What would
   happen in a grid search if `__init__` validated or transformed its arguments?

<details><summary><b>Show answers</b></summary>

1. From the signature of `__init__` (`inspect.signature(cls.__init__)`), sorted by name. That is why
   `*args` is forbidden: the names must be discoverable.
2. It reads `getattr(self, "threshold")`, but the object only has `cutoff`.
3. `GridSearchCV` creates clones and then calls `set_params(threshold=...)`, which sets the attribute
   directly and never runs `__init__` again. Any validation or transformation in `__init__` would be
   skipped for every value in the grid, and a clone built from transformed values would differ from
   the original. That is why scikit-learn validates parameters in `fit`. Notebook X2 builds estimators
   that follow these rules.
</details>

---

## How to read unfamiliar code: a checklist

1. **Start from what you call.** Find the public method you use (`fit`, `predict`, `__getitem__`) rather
   than reading a file top to bottom.
2. **Find where it is really defined.** Print `cls.__mro__`, or use *Go to Definition* (F12 in VS Code)
   on the method. It is often in a parent class or a mixin.
3. **Read `__init__` and `fit` first.** They tell you what state the object has and when it changes.
4. **Skip what you do not need.** Validation, metadata routing and deprecation code is important for the
   library but rarely for your question.
5. **Run small experiments** next to the code (`vars(obj)`, `type(...)`, tiny inputs).
6. **Explain it to someone** (or a rubber duck). If you cannot explain it, you have found the next
   thing to read.

## Summary

- Real libraries are built from the same pieces you used today: inheritance and `super()`, mixins,
  dunder methods, composition, and conventions such as "fit returns self" and "learned attributes end
  with an underscore".
- `inspect.getsource`, `__mro__`, `vars()` and *Go to Definition* let you check how things really work
  instead of guessing.
- Conventions are load-bearing: `check_is_fitted` and `get_params` depend on them. Follow them when you
  write estimators of your own.

## Going further

- Pick a class you use often (`pandas.DataFrame`, `torch.nn.Module`, `sklearn.model_selection.KFold`),
  print its MRO and read its `__init__`. Which of today's concepts do you find?

---

## Sources and further reading

**Sources.** This notebook is based on:

- Python docs: [`inspect`](https://docs.python.org/3/library/inspect.html) and [`pathlib`](https://docs.python.org/3/library/pathlib.html).
- [scikit-learn source code on GitHub](https://github.com/scikit-learn/scikit-learn/tree/main/sklearn), the files we read (`base.py`, `dummy.py`, `pipeline.py`, `utils/validation.py`).
- [scikit-learn glossary](https://scikit-learn.org/stable/glossary.html), the official meaning of terms such as estimator, fitted, clone and parameters.

**For self-study.** Where to go next if you want to learn more:

- [Buitinck et al. (2013), *API design for machine learning software: experiences from the scikit-learn project*](https://arxiv.org/abs/1309.0238) (paper, intermediate): the scikit-learn authors explain the design you just read: estimators, the fit/predict/transform interface, composition and why parameters live in `__init__`. Short and very readable.
- [scikit-learn: Developing scikit-learn estimators](https://scikit-learn.org/stable/developers/develop.html) (guide, advanced), the rules behind the code. Continue with notebook X2.

---

[← 05 · Polymorphism, duck typing and protocols](05_polymorphism_duck_typing_protocols.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [07 · Type hints, docstrings and dataclasses →](07_type_hints_docstrings_dataclasses.ipynb)